In [1]:
# ============================================================
# Notebook    : 15_case_a_vehtype_diagnosis.ipynb
# Description : Two-part follow-up to §4.7's unresolved pattern
#               (Case A, VehType proxy cohort, ③c model):
#
#               PART 1 — Bootstrap CI on calibration error
#               (predicted_positive_rate - actual_positive_rate)
#               per VehType, restricted to n>=100 groups. This
#               answers: "is the over/under-prediction pattern
#               statistically real, or could it be noise even in
#               the 'reliable' groups?"
#
#               PART 2 — Diagnosis of WHY the pattern exists:
#               is VehType genuinely this correlated with risk in
#               the raw data (legitimate), or does the model
#               amplify the correlation beyond what the raw data
#               shows (distortion)? Compared via:
#                 (a) raw actual-risk vs model-predicted-risk
#                     scatter per VehType
#                 (b) calibration curve stratified by VehType
#                     risk tier (low/mid/high, not all 16
#                     categories, to keep the curve readable)
# ============================================================


# ============================================================
# 0. Install dependencies (run once)
# ============================================================
# pip install lightgbm pandas numpy matplotlib


# ============================================================
# 1. Common imports
# ============================================================
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt

np.random.seed(42)
N_BOOTSTRAP = 1000


# ============================================================
# 2. Reconstruct the EXACT ③c test set — same as 03c/09/10/13
# ============================================================
df = pd.read_csv("data/fremotor_multi_history_aggregate.csv")

with open("data/sequences/split_idpols.json", "r", encoding="utf-8") as f:
    split_ids = json.load(f)
test_idpols = set(split_ids["test"])

df_test = df[df["IDpol"].isin(test_idpols)].copy()

NUMERIC_COLS = ["Expo", "YearGap", "CumClaimCount", "ClaimRateSoFar", "YearsSinceFirstClaimLog"]
BINARY_COLS  = ["PrevLabel", "HasPriorClaim"]
CAT_COLS     = ["Usage", "VehType", "VehPower"]
FEATURE_COLS = NUMERIC_COLS + BINARY_COLS + CAT_COLS

for col in CAT_COLS:
    df_test[col] = df_test[col].astype("category")

X_test = df_test[FEATURE_COLS].copy()
y_test = df_test["Label"].copy()

model = lgb.Booster(model_file="data/sequences/lightgbm_aggregate_model.txt")
y_pred_prob = model.predict(X_test)
y_pred_cls  = (y_pred_prob >= 0.5).astype(int)

print(f"[CHECK 1] Test set: {X_test.shape}, positive rate: {y_test.mean()*100:.2f}%")


# ============================================================
# 3. PART 1 — Bootstrap CI on calibration error per VehType
#    (n>=100 groups only, per §4.7's "reliable groups" definition)
# ============================================================
print("\n" + "="*60)
print("PART 1 — Bootstrap CI on calibration error (n>=100 groups)")
print("="*60)

vehtype_counts = df_test["VehType"].value_counts()
reliable_types = vehtype_counts[vehtype_counts >= 100].index.tolist()
print(f"[CHECK 2] Reliable VehType groups (n>=100): {len(reliable_types)}")
print(f"[CHECK 2] Groups: {sorted(reliable_types)}")

def bootstrap_calibration_error(y_true, y_pred_prob, mask, n_boot=N_BOOTSTRAP, seed=42):
    """
    calibration_error = predicted_positive_rate - actual_positive_rate,
    within the given group mask. Bootstrap resamples WITHIN the
    group (not across groups) to get a CI on this single group's
    calibration error.
    """
    yt = y_true[mask].values
    ypp = y_pred_prob[mask]
    n = len(yt)

    point_est = ypp.mean() - yt.mean()  # predicted - actual, at thr=0.5 implicit via mean prob... 
    # NOTE: using mean predicted PROBABILITY (not thresholded class)
    # as "predicted positive rate" here, consistent with how
    # over/under-prediction was framed in §4.7 (mean_pred_prob vs
    # actual_positive_rate), not the thr=0.5 classification rate.

    rng = np.random.RandomState(seed)
    boot_errors = np.zeros(n_boot)
    for b in range(n_boot):
        idx = rng.randint(0, n, size=n)
        boot_errors[b] = ypp[idx].mean() - yt[idx].mean()

    ci_low, ci_high = np.percentile(boot_errors, [2.5, 97.5])
    significant = not (ci_low <= 0 <= ci_high)
    return point_est, ci_low, ci_high, significant, n

calib_results = []
for vt in sorted(reliable_types):
    mask = (df_test["VehType"] == vt).values
    point_est, ci_low, ci_high, sig, n = bootstrap_calibration_error(y_test, y_pred_prob, mask)
    calib_results.append({
        "VehType": vt, "n": n,
        "actual_positive_rate": y_test[mask].mean(),
        "mean_pred_prob": y_pred_prob[mask].mean(),
        "calibration_error": point_est,
        "ci_low": ci_low, "ci_high": ci_high,
        "significant_miscalibration": sig,
    })

calib_df = pd.DataFrame(calib_results).sort_values("calibration_error")
print("\n[PART 1] Calibration error by VehType (predicted - actual), with 95% bootstrap CI:")
print(calib_df.to_string(index=False))

n_sig = calib_df["significant_miscalibration"].sum()
print(f"\n[PART 1] {n_sig}/{len(calib_df)} reliable VehType groups show statistically "
      f"significant miscalibration (95% CI excludes 0)")

sig_groups = calib_df[calib_df["significant_miscalibration"]]["VehType"].tolist()
print(f"[PART 1] Significantly miscalibrated groups: {sig_groups}")

calib_df.to_csv("paper_assets/tables/Table6_CaseA_VehType_Calibration_CI.csv", index=False)
print("Saved: paper_assets/tables/Table6_CaseA_VehType_Calibration_CI.csv")


# ============================================================
# 4. Visualization — calibration error with CI, sorted
# ============================================================
fig, ax = plt.subplots(figsize=(10, 6))
colors = ["tab:red" if sig else "tab:blue" for sig in calib_df["significant_miscalibration"]]
y_pos = np.arange(len(calib_df))
ax.barh(y_pos, calib_df["calibration_error"], color=colors, alpha=0.7)
ax.errorbar(calib_df["calibration_error"], y_pos,
            xerr=[calib_df["calibration_error"] - calib_df["ci_low"],
                  calib_df["ci_high"] - calib_df["calibration_error"]],
            fmt="none", ecolor="black", capsize=3)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(calib_df["VehType"])
ax.set_xlabel("Calibration error (mean predicted prob - actual positive rate)")
ax.set_title("Case A (③c) — VehType calibration error with 95% bootstrap CI\n(red = statistically significant miscalibration, n>=100 only)")
plt.tight_layout()
plt.savefig("paper_assets/figures/Fig3_CaseA_VehType_Calibration_CI.png", dpi=150)
plt.close()
print("\nSaved: paper_assets/figures/Fig3_CaseA_VehType_Calibration_CI.png")


# ============================================================
# 5. PART 2 — Diagnosis: legitimate risk signal vs model
#    distortion. Two complementary views.
# ============================================================
print("\n" + "="*60)
print("PART 2 — Diagnosis: legitimate signal vs model distortion")
print("="*60)

# ------------------------------------------------------------
# 5a. Raw actual-risk vs model-predicted-risk scatter, all
#     reliable VehType groups. If points fall near the y=x line,
#     the model is faithfully reflecting real risk differences
#     (legitimate). If points deviate systematically (e.g. high-
#     risk groups pushed higher, low-risk groups pushed lower
#     than actual), that's amplification/distortion.
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(calib_df["actual_positive_rate"], calib_df["mean_pred_prob"],
           s=calib_df["n"] / 100, alpha=0.6, c=colors)
lims = [0, max(calib_df["actual_positive_rate"].max(), calib_df["mean_pred_prob"].max()) * 1.1]
ax.plot(lims, lims, "k--", linewidth=1, label="y = x (perfect calibration)")
for _, row in calib_df.iterrows():
    ax.annotate(row["VehType"], (row["actual_positive_rate"], row["mean_pred_prob"]),
                fontsize=7, alpha=0.7)
ax.set_xlabel("Actual positive rate (raw data)")
ax.set_ylabel("Mean predicted probability (model)")
ax.set_title("Case A — Actual risk vs predicted risk by VehType\n(point size ~ n, red = significantly miscalibrated)")
ax.legend()
plt.tight_layout()
plt.savefig("paper_assets/figures/Fig4_CaseA_VehType_ActualVsPredicted.png", dpi=150)
plt.close()
print("Saved: paper_assets/figures/Fig4_CaseA_VehType_ActualVsPredicted.png")

# quantify: is there systematic amplification? regress
# mean_pred_prob on actual_positive_rate — slope > 1 means the
# model amplifies risk differences beyond what the raw data shows
from numpy.polynomial import polynomial as P
slope, intercept = np.polyfit(calib_df["actual_positive_rate"], calib_df["mean_pred_prob"], 1)
print(f"\n[PART 2a] Regression: predicted = {slope:.3f} * actual + {intercept:.4f}")
if slope > 1.1:
    print(f"[PART 2a] Slope > 1.1 — model AMPLIFIES actual risk differences across VehType")
elif slope < 0.9:
    print(f"[PART 2a] Slope < 0.9 — model COMPRESSES actual risk differences across VehType")
else:
    print(f"[PART 2a] Slope near 1.0 — model roughly PRESERVES actual risk differences (legitimate reflection)")


# ------------------------------------------------------------
# 5b. Calibration curve stratified by VehType RISK TIER
#     (low/mid/high actual risk, not all 16 categories) — shows
#     whether miscalibration concentrates at specific risk levels
#     regardless of which VehType, or is VehType-specific.
# ------------------------------------------------------------
tier_cutoffs = df_test["VehType"].map(
    dict(zip(calib_df["VehType"], calib_df["actual_positive_rate"]))
)
df_test["_actual_risk_for_tier"] = tier_cutoffs

low_thr, high_thr = df_test["_actual_risk_for_tier"].quantile([0.33, 0.67])
def assign_tier(r):
    if pd.isna(r):
        return "excluded (small-n VehType)"
    elif r <= low_thr:
        return "Low actual risk"
    elif r <= high_thr:
        return "Mid actual risk"
    else:
        return "High actual risk"

df_test["risk_tier"] = df_test["_actual_risk_for_tier"].apply(assign_tier)

print(f"\n[PART 2b] Risk tier boundaries: low<={low_thr:.4f}, high>{high_thr:.4f}")
print(df_test["risk_tier"].value_counts())

fig, ax = plt.subplots(figsize=(8, 6))
for tier in ["Low actual risk", "Mid actual risk", "High actual risk"]:
    tier_mask = (df_test["risk_tier"] == tier).values
    if tier_mask.sum() < 50:
        continue
    yt_tier = y_test[tier_mask].values
    ypp_tier = y_pred_prob[tier_mask]
    bins = pd.qcut(ypp_tier, min(5, len(np.unique(ypp_tier))), duplicates="drop")
    tier_df = pd.DataFrame({"y": yt_tier, "p": ypp_tier, "bin": bins})
    binned = tier_df.groupby("bin", observed=True).agg(
        mean_pred=("p", "mean"), actual_rate=("y", "mean"), n=("y", "size")
    ).reset_index()
    ax.plot(binned["mean_pred"], binned["actual_rate"], marker="o", label=tier)

ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Perfect calibration")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Actual positive rate")
ax.set_title("Case A — Calibration curve by VehType actual-risk tier")
ax.legend()
plt.tight_layout()
plt.savefig("paper_assets/figures/Fig5_CaseA_CalibrationByRiskTier.png", dpi=150)
plt.close()
print("\nSaved: paper_assets/figures/Fig5_CaseA_CalibrationByRiskTier.png")


# ============================================================
# 6. Summary and interpretation
# ============================================================
print("\n===== §6.9 Follow-up: Summary =====")
print(f"Reliable VehType groups tested: {len(calib_df)}")
print(f"Significantly miscalibrated (95% CI excludes 0): {n_sig} — {sig_groups}")
print(f"Regression slope (predicted ~ actual): {slope:.3f}")
print(f"\nInterpretation guide:")
print(f"  - If n_sig is small AND slope ~ 1.0: pattern is mostly noise/legitimate,")
print(f"    §4.7's caution can be softened.")
print(f"  - If n_sig is large AND slope > 1.1: model amplifies real risk gaps —")
print(f"    genuine distortion, §4.7's concern is confirmed and should be")
print(f"    elevated from 'unresolved' to 'confirmed limitation'.")
print(f"  - If n_sig is large BUT slope ~ 1.0: individual groups miscalibrated")
print(f"    but no systematic direction — different, more localized issue.")
print("=========================================")

[CHECK 1] Test set: (54755, 10), positive rate: 12.79%

PART 1 — Bootstrap CI on calibration error (n>=100 groups)
[CHECK 2] Reliable VehType groups (n>=100): 10
[CHECK 2] Groups: ['VehType10', 'VehType11', 'VehType12', 'VehType13', 'VehType3', 'VehType5', 'VehType6', 'VehType7', 'VehType8', 'VehType9']

[PART 1] Calibration error by VehType (predicted - actual), with 95% bootstrap CI:
  VehType     n  actual_positive_rate  mean_pred_prob  calibration_error   ci_low  ci_high  significant_miscalibration
VehType13   120              0.025000        0.102931           0.077931 0.047658 0.103464                        True
 VehType8  3545              0.015797        0.098848           0.083052 0.078369 0.087403                        True
 VehType9  1922              0.022893        0.117848           0.094955 0.087643 0.101791                        True
VehType12   843              0.060498        0.256740           0.196242 0.177409 0.214766                        True
 VehType7  1328 

In [2]:
# ============================================================
# 7. Global calibration check — is the +bias seen in EVERY
#    VehType group actually a single global bias, not a
#    VehType-specific pattern?
# ============================================================
print("\n" + "="*60)
print("PART 3 — Global calibration check")
print("="*60)

global_actual_rate = y_test.mean()
global_mean_pred_prob = y_pred_prob.mean()
global_calib_error = global_mean_pred_prob - global_actual_rate

print(f"[GLOBAL] Actual positive rate (full test set): {global_actual_rate:.4f}")
print(f"[GLOBAL] Mean predicted probability (full test set): {global_mean_pred_prob:.4f}")
print(f"[GLOBAL] Global calibration error: {global_calib_error:+.4f}")

# bootstrap CI on the global error, same method as per-group
rng = np.random.RandomState(42)
n_full = len(y_test)
y_test_arr = y_test.values
global_boot_errors = np.zeros(N_BOOTSTRAP)
for b in range(N_BOOTSTRAP):
    idx = rng.randint(0, n_full, size=n_full)
    global_boot_errors[b] = y_pred_prob[idx].mean() - y_test_arr[idx].mean()
global_ci_low, global_ci_high = np.percentile(global_boot_errors, [2.5, 97.5])
print(f"[GLOBAL] 95% CI: [{global_ci_low:+.4f}, {global_ci_high:+.4f}]")
print(f"[GLOBAL] Significant global miscalibration: {not (global_ci_low <= 0 <= global_ci_high)}")

# ------------------------------------------------------------
# Decompose: how much of each group's calibration error is
# "just the global error" vs "extra, group-specific error"?
# ------------------------------------------------------------
calib_df["global_calib_error"] = global_calib_error
calib_df["excess_calib_error"] = calib_df["calibration_error"] - global_calib_error

print(f"\n[DECOMPOSITION] Each group's error minus the global error "
      f"(= {global_calib_error:+.4f}):")
print(calib_df[["VehType", "n", "calibration_error", "excess_calib_error"]].to_string(index=False))

# re-test significance of the EXCESS error (group error beyond
# what global bias alone would predict), via bootstrap of the
# DIFFERENCE between group and global error
def bootstrap_excess_error(y_true, y_pred_prob, group_mask, n_boot=N_BOOTSTRAP, seed=42):
    yt_g = y_true[group_mask].values
    ypp_g = y_pred_prob[group_mask]
    n_g = len(yt_g)
    yt_all = y_true.values
    ypp_all = y_pred_prob
    n_all = len(yt_all)

    rng = np.random.RandomState(seed)
    excess = np.zeros(n_boot)
    for b in range(n_boot):
        idx_g = rng.randint(0, n_g, size=n_g)
        idx_all = rng.randint(0, n_all, size=n_all)
        group_err = ypp_g[idx_g].mean() - yt_g[idx_g].mean()
        global_err = ypp_all[idx_all].mean() - yt_all[idx_all].mean()
        excess[b] = group_err - global_err

    ci_low, ci_high = np.percentile(excess, [2.5, 97.5])
    return not (ci_low <= 0 <= ci_high), ci_low, ci_high

excess_results = []
for _, row in calib_df.iterrows():
    mask = (df_test["VehType"] == row["VehType"]).values
    sig_excess, exc_lo, exc_hi = bootstrap_excess_error(y_test, y_pred_prob, mask)
    excess_results.append({"VehType": row["VehType"], "excess_significant": sig_excess,
                            "excess_ci_low": exc_lo, "excess_ci_high": exc_hi})

excess_df = pd.DataFrame(excess_results)
calib_df_final = calib_df.merge(excess_df, on="VehType")

n_excess_sig = calib_df_final["excess_significant"].sum()
print(f"\n[DECOMPOSITION] After removing global bias, groups still significantly "
      f"miscalibrated: {n_excess_sig}/{len(calib_df_final)}")
print(calib_df_final[["VehType", "n", "excess_calib_error", "excess_significant"]].to_string(index=False))

calib_df_final.to_csv("paper_assets/tables/Table6_CaseA_VehType_Calibration_CI.csv", index=False)
print("\nSaved (updated): paper_assets/tables/Table6_CaseA_VehType_Calibration_CI.csv")


# ============================================================
# 8. Revised interpretation
# ============================================================
print("\n===== REVISED Interpretation =====")
print(f"Global calibration error: {global_calib_error:+.4f} "
      f"(significant: {not (global_ci_low <= 0 <= global_ci_high)})")
print(f"Groups significant BEFORE removing global bias: {n_sig}/{len(calib_df)}")
print(f"Groups significant AFTER removing global bias (excess error): {n_excess_sig}/{len(calib_df_final)}")
if n_excess_sig < n_sig:
    print(f"\n=> The pattern is PARTIALLY explained by a global upward bias")
    print(f"   (likely from scale_pos_weight class-imbalance correction).")
    print(f"   {n_sig - n_excess_sig} group(s) were significant only because of")
    print(f"   the global shift, not a VehType-specific issue.")
if n_excess_sig > 0:
    print(f"\n=> {n_excess_sig} group(s) remain significantly miscalibrated EVEN")
    print(f"   AFTER accounting for global bias — this is genuine VehType-specific")
    print(f"   distortion, not just the model's overall calibration offset.")
print("===================================")


PART 3 — Global calibration check
[GLOBAL] Actual positive rate (full test set): 0.1279
[GLOBAL] Mean predicted probability (full test set): 0.3987
[GLOBAL] Global calibration error: +0.2708
[GLOBAL] 95% CI: [+0.2675, +0.2737]
[GLOBAL] Significant global miscalibration: True

[DECOMPOSITION] Each group's error minus the global error (= +0.2708):
  VehType     n  calibration_error  excess_calib_error
VehType13   120           0.077931           -0.192827
 VehType8  3545           0.083052           -0.187706
 VehType9  1922           0.094955           -0.175803
VehType12   843           0.196242           -0.074516
 VehType7  1328           0.198582           -0.072176
VehType11  2635           0.232136           -0.038622
 VehType5   220           0.281502            0.010744
 VehType6  3118           0.292508            0.021750
VehType10 39315           0.300197            0.029439
 VehType3  1550           0.317709            0.046951

[DECOMPOSITION] After removing global bias, g